In [ ]:
# Music For Robots (M4R) — Experiment G / M4R-07: Turn-0 Flood Creative Brainstorming
# Grounded in: Alternative Uses Task (AUT), Divergent Association Task (DAT, Olson et al. 2021),
# and dense semantic embedding dispersion (sentence-transformers all-MiniLM-L6-v2).
#
# 4 Turn-0 Conditions x 3 Seeds (11, 22, 33) x 5 Brainstorming Tasks = 60 Evaluations (+ 9 Turn-0 wakes):
#   C0_ColdBoot                 : No audio on Turn 0
#   C1_ShortLock_9s             : 9.25s Ties That Bide on Turn 0
#   C2_StutteringPackets_200s   : 199.9s Stuttering Packets on Turn 0 (deliberate buffer overflow!)
#   C3_ClickOverflow_200s       : 199.9s 120 BPM mechanical click on Turn 0 (length-matched control)

import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers", "bitsandbytes", "accelerate", "librosa", "soundfile",
    "scikit-learn", "sentence-transformers"
])

import json, re, time
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import torch
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration, BitsAndBytesConfig

DATA_ROOT = Path("/tmp/m4r-data")
if not (DATA_ROOT / "tracks" / "stuttering-packets").exists():
    if DATA_ROOT.exists():
        subprocess.check_call(["rm", "-rf", str(DATA_ROOT)])
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/luchtan-GIT/m4r-data.git", str(DATA_ROOT)])

MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
SR = processor.feature_extractor.sampling_rate

y_short, _ = librosa.load(str(DATA_ROOT / "tracks" / "ties-that-bide" / "audio.mp3"), sr=SR, mono=True)
y_flood, _ = librosa.load(str(DATA_ROOT / "tracks" / "stuttering-packets" / "audio.mp3"), sr=SR, mono=True)

def make_click_track(n_samples: int, bpm: float = 120.0, sr: int = SR):
    y = np.zeros(n_samples, dtype=np.float32)
    step = int((60.0 / bpm) * sr)
    click_len = int(0.015 * sr)
    t = np.arange(click_len) / sr
    ping = (0.35 * np.sin(2 * np.pi * 880 * t) * np.exp(-t * 140)).astype(np.float32)
    for pos in range(0, n_samples - click_len, step):
        y[pos:pos + click_len] += ping
    return y

y_click = make_click_track(len(y_flood), bpm=120.0, sr=SR)

CONDITIONS = {
    "C0_ColdBoot": None,
    "C1_ShortLock_9s": y_short.astype(np.float32),
    "C2_StutteringPackets_200s": y_flood.astype(np.float32),
    "C3_ClickOverflow_200s": y_click.astype(np.float32),
}

TASKS = [
    ("AUT_brick", 8, "List 8 unusual, non-obvious uses for a brick. Format as a numbered list 1-8, one concise sentence each."),
    ("AUT_paperclip", 8, "List 8 unusual, non-obvious uses for a paperclip. Format as a numbered list 1-8, one concise sentence each."),
    ("DAT_10_nouns", 10, "Name 10 single English nouns that are as unrelated to each other in meaning as possible. Output only a numbered list 1-10 of single nouns."),
    ("IDEATION_subway", 6, "Brainstorm 6 unconventional ways an empty subway station at 3 AM could be used without spending money. Format as a numbered list 1-6, one concise sentence each."),
    ("INVENTION_time", 6, "Invent 6 distinct mechanisms for telling time without numbers, hands, or screens. Format as a numbered list 1-6, one concise sentence each."),
]

CLICHE_TERMS = {
    "AUT_brick": ["doorstop", "door stop", "paperweight", "paper weight", "weapon", "hammer", "bookend", "weight", "step"],
    "AUT_paperclip": ["bookmark", "book mark", "sim", "reset", "lockpick", "lock pick", "zipper", "hairpin", "earring", "toothpick", "hook"],
}

SEEDS = [11, 22, 33]

def generate_reply(conversation, audios, seed=11, max_new_tokens=260):
    torch.manual_seed(seed)
    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    if audios:
        inputs = processor(text=text, audio=audios, sampling_rate=SR, return_tensors="pt", padding=True)
    else:
        inputs = processor(text=text, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.92,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    gen_ids = out_ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(gen_ids, skip_special_tokens=True)[0].strip()

def parse_numbered_items(text: str):
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    items = []
    for ln in lines:
        m = re.match(r"^(?:\d+[\.\)\:\-]\s*|\-\s*|\*\s*)(.+)$", ln)
        if m:
            items.append(m.group(1).strip())
    if len(items) < 3:
        items = [s.strip() for s in re.split(r"[\n;]+", text) if len(s.strip()) > 2]
    return items

rows = []
transcripts = ["# Experiment G (M4R-07) — Turn-0 Flood Creative Brainstorming (AUT + DAT)\n"]

for cond_name, y_audio in CONDITIONS.items():
    print(f"\n=== CONDITION: {cond_name} ===")
    for seed in SEEDS:
        conv = []
        audios = []
        if y_audio is not None:
            audios = [y_audio]
            conv.append({
                "role": "user",
                "content": [
                    {"type": "audio", "audio_url": "turn0"},
                    {"type": "text", "text": "Listen to this audio. You do not need to analyze or summarize it — simply take it in and say 'Ready.' when it finishes."},
                ],
            })
            wake_rep = generate_reply(conv, audios, seed=seed, max_new_tokens=30)
            conv.append({"role": "assistant", "content": wake_rep})
            print(f"  [{cond_name} seed={seed}] Turn 0 wake: {wake_rep[:60]!r}")

        if seed == 11:
            transcripts.append(f"## `{cond_name}` (Seed 11)\n")

        for t_idx, (task_id, target_n, prompt) in enumerate(TASKS, start=1):
            # Fresh task turn right after Turn-0 wake (keeps each brainstorming task clean & independent!)
            task_conv = list(conv) + [{"role": "user", "content": prompt}]
            rep = generate_reply(task_conv, audios, seed=seed + t_idx * 17, max_new_tokens=260)
            items = parse_numbered_items(rep)
            low = rep.lower()
            cliches = sum(1 for c in CLICHE_TERMS.get(task_id, []) if c in low)
            rows.append({
                "condition": cond_name,
                "seed": seed,
                "task_id": task_id,
                "target_n": target_n,
                "n_items": len(items),
                "format_hit": int(len(items) >= target_n),
                "cliche_hits": cliches,
                "items_json": json.dumps(items),
                "response": rep.replace("\n", " | "),
            })
            if seed == 11:
                transcripts.append(f"### `{task_id}`\n{rep}\n")
            print(f"  [{cond_name} seed={seed} {task_id}] items={len(items)}/{target_n} cliches={cliches}")

# Free Qwen2-Audio VRAM before loading sentence-transformers on CPU/GPU
del model
torch.cuda.empty_cache()

embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
df = pd.DataFrame(rows)

# 1. Intra-session semantic dispersion (mean pairwise cosine distance among items inside each response)
intra_disp = []
for _, r in df.iterrows():
    items = json.loads(r["items_json"])
    if len(items) >= 2:
        embs = embedder.encode(items, normalize_embeddings=True)
        sim = cosine_similarity(embs)
        tri = sim[np.triu_indices(len(items), k=1)]
        intra_disp.append(round(float(1.0 - np.mean(tri)), 4))
    else:
        intra_disp.append(0.0)
df["intra_semantic_dispersion"] = intra_disp

# 2. Full-response embeddings for Divergence from C0 & Cross-Seed Originality
full_embs = embedder.encode(df["response"].tolist(), normalize_embeddings=True)

div_c0 = []
for idx, r in df.iterrows():
    if r["condition"] == "C0_ColdBoot":
        div_c0.append(0.0)
        continue
    base = df[(df["condition"] == "C0_ColdBoot") & (df["seed"] == r["seed"]) & (df["task_id"] == r["task_id"])]
    if len(base) > 0:
        b_idx = base.index[0]
        sim = float(np.dot(full_embs[idx], full_embs[b_idx]))
        div_c0.append(round(1.0 - sim, 4))
    else:
        div_c0.append(0.0)
df["semantic_div_from_c0"] = div_c0

# Cross-seed originality (1 - mean pairwise cosine sim across seeds for same condition & task)
cross_seed_orig = {}
for cond_name in CONDITIONS:
    dists = []
    for task_id, _, _ in TASKS:
        idxs = df[(df["condition"] == cond_name) & (df["task_id"] == task_id)].index.tolist()
        if len(idxs) >= 2:
            mat = cosine_similarity(full_embs[idxs])
            tri = mat[np.triu_indices(len(idxs), k=1)]
            dists.extend((1.0 - tri).tolist())
    cross_seed_orig[cond_name] = round(float(np.mean(dists)), 4) if dists else 0.0

df.to_csv("/kaggle/working/experiment_g_results.csv", index=False)
Path("/kaggle/working/experiment_g_transcripts.md").write_text("\n".join(transcripts), encoding="utf-8")

summary = ["# Experiment G (M4R-07) Summary — Turn-0 Flood Creative Brainstorming (AUT + DAT)\n"]
summary.append("## Overall Comparison Across 5 Brainstorming Tasks (15 Trials / Condition, all-MiniLM-L6-v2 Embeddings)\n")
summary.append("| Turn-0 Condition | Intra-List Semantic Dispersion | Cross-Seed Originality (Anti-Collapse) | Semantic Div from C0 | AUT Cliché Hits | Format Compliance |")
summary.append("|---|---:|---:|---:|---:|---:|")
for cond_name in CONDITIONS:
    g = df[df["condition"] == cond_name]
    summary.append(
        f"| `{cond_name}` | {g['intra_semantic_dispersion'].mean():.4f} | {cross_seed_orig[cond_name]:.4f} | "
        f"{g['semantic_div_from_c0'].mean():.4f} | {int(g['cliche_hits'].sum())} | {g['format_hit'].mean()*100:.1f}% |"
    )

summary.append("\n## Per-Task Intra-List Semantic Dispersion (Higher = Broader Conceptual Range)\n")
summary.append("| Turn-0 Condition | AUT_brick | AUT_paperclip | DAT_10_nouns | IDEATION_subway | INVENTION_time |")
summary.append("|---|---:|---:|---:|---:|---:|")
for cond_name in CONDITIONS:
    vals = [f"{df[(df['condition']==cond_name) & (df['task_id']==t)]['intra_semantic_dispersion'].mean():.4f}" for t, _, _ in TASKS]
    summary.append(f"| `{cond_name}` | " + " | ".join(vals) + " |")

Path("/kaggle/working/experiment_g_summary.md").write_text("\n".join(summary), encoding="utf-8")
print("\nDONE! Total brainstorming evaluations:", len(df))
print("\n".join(summary))
